In [40]:
import os
import sys

sys.path.append("../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [41]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data,
)
from utils.model_utils.load_model import load_model
from utils.model_utils.save_module import save_module
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.sampling import SamplingDataset

In [42]:
name = "bert-4-128-yahoo"
# name = "YahooAnswersTopics"
device = torch.device("cuda:0")
checkpoint = None
batch_size = 16
num_workers = 4
num_samples = 128
head_pruning_ratio = 0.5
seed = 44
include_layers = ["attention", "intermediate", "output"]
exclude_layers = None

In [43]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.
{'model_name': 'Models/bert-4-128-yahoo', 'tokenizer_name': 'sadickam/sdg-classification-bert', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}
The model Models/bert-4-128-yahoo is loaded.


In [44]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    model_config.dataset_name,
    batch_size=batch_size,
    num_workers=num_workers,
    do_cache=True,
    seed=seed,
)

Loading cached dataset YahooAnswersTopics.
train.pkl is loaded from cache.
valid.pkl is loaded from cache.
test.pkl is loaded from cache.
The dataset YahooAnswersTopics is loaded
{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/YahooAnswersTopics', 'task_type': 'classification'}


In [45]:
import numpy as np
import torch
import copy
from functools import partial
import torch.nn as nn
from transformers.pytorch_utils import find_pruneable_heads_and_indices
from typing import *

import time


def calculate_prune_head(arr, i, pruned_heads):
    flattened_with_indices = [
        (value, index)
        for index, value in np.ndenumerate(arr)
        if index not in pruned_heads
    ]

    sorted_by_value = sorted(flattened_with_indices, key=lambda x: x[0])
    bottom_indices = sorted_by_value[:i]

    bottom_indices_only = [index for _, index in bottom_indices]

    return bottom_indices_only


def prune_head(model, prune_list):
    for layer_index, head_index in prune_list:
        prune_heads(model.bert.encoder.layer[layer_index].attention, ([head_index]))
    return model


def preprocess_prunehead(arr, num_layer):
    layer_max = lambda arr: np.argmax(arr, axis=1)

    max_layer = layer_max(arr)
    for layer in range(num_layer):
        head = max_layer[layer]
        arr[layer][head] = 100
    return arr


def prune_heads(layer, heads):
    if len(heads) == 0:
        return
    heads, index = find_pruneable_heads_and_indices(
        heads,
        layer.self.num_attention_heads,
        layer.self.attention_head_size,
        layer.pruned_heads,
    )

    # Zero out weights in linear layers instead of pruning
    layer.self.query = zero_out_head_weights(
        layer.self.query, heads, layer.self.attention_head_size
    )
    layer.self.key = zero_out_head_weights(
        layer.self.key, heads, layer.self.attention_head_size
    )
    layer.self.value = zero_out_head_weights(
        layer.self.value, heads, layer.self.attention_head_size
    )
    layer.output.dense = zero_out_head_weights(
        layer.output.dense, heads, layer.self.attention_head_size, dim=1
    )


def zero_out_head_weights(
    layer: nn.Linear, heads: Set[int], head_size: int, dim: int = 0
) -> nn.Linear:
    """
    Zero out the weights of the specified heads in the linear layer.

    Args:
        layer (`torch.nn.Linear`): The layer to modify.
        heads (`Set[int]`): The indices of heads to zero out.
        head_size (`int`): The size of each head.
        dim (`int`, *optional*, defaults to 0): The dimension on which to zero out the weights.

    Returns:
        `torch.nn.Linear`: The modified layer with weights of specified heads zeroed out.
    """
    for head in heads:
        start_index = head * head_size
        end_index = (head + 1) * head_size
        if dim == 0:
            layer.weight.data[start_index:end_index] = 0
        elif dim == 1:
            layer.weight.data[:, start_index:end_index] = 0

    return layer

In [46]:
import numpy as np


def calculate_head_importance(
    model,
    model_config,
    data,
    normalize_scores_by_layer=True,
):
    device = model_config.device
    from functools import partial

    gradients = {}
    context_layers = {}

    def save_grad(gradients, layer_idx, grad):
        gradients[f"context_layer_{layer_idx}"] = grad

    def forward_hook(module, input, output, gradients, context_layers, layer_idx):
        context_layers[f"context_layer_{layer_idx}"] = output[0]
        output[0].register_hook(partial(save_grad, gradients, layer_idx))

    def reshape(tensors, shape, num_heads):
        batch_size = shape[0]
        seq_len = shape[1]
        head_dim = shape[2] // num_heads
        tensors = tensors.reshape(batch_size, seq_len, num_heads, head_dim)
        tensors = tensors.permute(0, 2, 1, 3)
        return tensors

    forward_handles = []

    for layer_idx in range(model.bert.config.num_hidden_layers):
        self_att = model.bert.encoder.layer[layer_idx].attention.self
        handle = self_att.register_forward_hook(
            partial(
                forward_hook,
                gradients=gradients,
                context_layers=context_layers,
                layer_idx=layer_idx,
            )
        )
        forward_handles.append(handle)

    # Disable dropout
    model.eval()
    device = device or next(model.parameters()).device

    n_layers = model.bert.config.num_hidden_layers
    n_heads = model.bert.config.num_attention_heads
    head_dim = model.bert.config.hidden_size // n_heads
    num_classes = model.num_labels  # Adjust based on your model

    head_importance = {
        label: torch.zeros(n_layers, n_heads).to(device) for label in range(num_classes)
    }
    tot_tokens = {label: 0 for label in range(num_classes)}

    for step, batch in enumerate(data):
        input_ids = batch["input_ids"].to(device)
        input_mask = batch["attention_mask"].to(device)
        label_ids = batch["labels"].to(device)
        unique_labels = label_ids.unique()

        for label in unique_labels:
            mask = label_ids == label
            input_ids_label = input_ids[mask]
            input_mask_label = input_mask[mask]
            label_ids_label = label_ids[mask]

            if input_ids_label.size(0) == 0:
                continue

            # Zero gradients
            model.zero_grad()
            # Compute loss and backward pass
            loss = model(
                input_ids_label, attention_mask=input_mask_label, labels=label_ids_label
            ).loss
            loss.backward()

            for layer_idx in range(n_layers):
                ctx = context_layers[f"context_layer_{layer_idx}"]
                grad_ctx = gradients[f"context_layer_{layer_idx}"]
                shape = ctx.shape
                ctx = reshape(ctx, shape, n_heads)
                grad_ctx = reshape(grad_ctx, shape, n_heads)

                # Compute dot product and accumulate head importance per class
                dot = torch.einsum("bhli,bhli->bhl", [grad_ctx, ctx])
                head_importance[label.item()][layer_idx] += (
                    dot.abs().sum(-1).sum(0).detach()
                )
                del ctx, grad_ctx, dot

            tot_tokens[label.item()] += input_mask_label.float().detach().sum().item()

    for label in range(num_classes):
        # Adjust the value weight norm addition
        for layer_idx in range(n_layers):
            for head in range(n_heads):
                start_idx = head * head_dim
                end_idx = (head + 1) * head_dim

                value_weight_norm = torch.norm(
                    model.bert.encoder.layer[layer_idx].attention.self.value.weight[
                        :, start_idx:end_idx
                    ]
                )
                head_importance[label][layer_idx][head] += value_weight_norm.detach()

        # Normalize head importance per class
        head_importance[label][:-1] /= (
            tot_tokens[label] + 1e-20
        )  # Avoid division by zero

        if normalize_scores_by_layer:
            exponent = 2
            norm_by_layer = torch.pow(
                torch.pow(head_importance[label], exponent).sum(-1), 1 / exponent
            )
            head_importance[label] /= norm_by_layer.unsqueeze(-1) + 1e-20

    for handle in forward_handles:
        handle.remove()

    return head_importance

In [47]:
positive_samples = SamplingDataset(
    train_dataloader,
    0,
    num_samples,
    num_labels,
    True,
    4,
    device=device,
    resample=False,
    seed=seed,
)
all_samples = SamplingDataset(
    train_dataloader,
    200,
    num_samples,
    num_labels,
    False,
    4,
    device=device,
    resample=False,
    seed=seed,
)

In [48]:
def head_importance_prunning(
    model, model_config, dominant_concern, concern, sparsity_ratio, gradually=True
):
    num_attention_heads = model.config.num_attention_heads
    num_hidden_layers = model.config.num_hidden_layers
    total_heads_to_prune = int(num_attention_heads * num_hidden_layers * sparsity_ratio)

    if total_heads_to_prune >= 4 and total_heads_to_prune % 4 != 0:
        total_heads_to_prune -= 4 - (total_heads_to_prune % 4)

    if gradually:
        num_steps = max(1, total_heads_to_prune // 4)
    else:
        num_steps = 1

    heads_per_step = int(total_heads_to_prune // num_steps)
    print(f"Total heads to prune: {total_heads_to_prune}")

    pruned_heads = set()

    for step in range(num_steps):
        if step == num_steps - 1:
            current_heads_to_prune = total_heads_to_prune - (step * heads_per_step)
        else:
            current_heads_to_prune = heads_per_step

        head_importance_list = calculate_head_importance(
            model,
            model_config,
            dominant_concern,
            normalize_scores_by_layer=False,
        )
        head_importance_list = head_importance_list[concern]
        print(f"head importance list\n {head_importance_list}")
        head_importance_list = head_importance_list.cpu()

        # preprocess_prunehead(head_importance_list, num_hidden_layers)

        prune_list = calculate_prune_head(
            head_importance_list, current_heads_to_prune, pruned_heads
        )
        pruned_heads.update(prune_list)

        prune_head(model, prune_list)
    print(pruned_heads)

In [49]:
module = copy.deepcopy(model)
head_importance_prunning(module, model_config, all_samples, 0, head_pruning_ratio)
result = evaluate_model(module, model_config, test_dataloader)
similar(
    model,
    module,
    valid_dataloader,
    0,
    num_samples,
    num_labels,
    device=device,
    seed=seed,
)

Total heads to prune: 8
head importance list
 tensor([[0.0112, 0.0111, 0.0118, 0.0117],
        [0.0113, 0.0121, 0.0119, 0.0115],
        [0.0104, 0.0114, 0.0105, 0.0111],
        [2.5688, 2.4918, 2.2326, 2.4640]], device='cuda:0')
head importance list
 tensor([[0.0101, 0.0067, 0.0108, 0.0107],
        [0.0110, 0.0119, 0.0117, 0.0112],
        [0.0039, 0.0077, 0.0041, 0.0040],
        [2.5677, 2.7412, 2.2992, 2.5731]], device='cuda:0')
{(0, 1), (2, 1), (0, 0), (0, 3), (2, 0), (2, 3), (0, 2), (2, 2)}


Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.3983
Precision: 0.6309, Recall: 0.5684, F1-Score: 0.5812
              precision    recall  f1-score   support

           0       0.43      0.58      0.50      2941
           1       0.66      0.40      0.50      2997
           2       0.70      0.59      0.64      3016
           3       0.29      0.65      0.40      2978
           4       0.82      0.56      0.67      3017
           5       0.84      0.70      0.76      3004
           6       0.64      0.39      0.48      3037
           7       0.53      0.57      0.55      3026
           8       0.67      0.58      0.62      2997
           9       0.72      0.65      0.68      2987

    accuracy                           0.57     30000
   macro avg       0.63      0.57      0.58     30000
weighted avg       0.63      0.57      0.58     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

In [50]:
for concern in range(num_labels):
    train = copy.deepcopy(train_dataloader)
    positive_samples = SamplingDataset(
        train,
        concern,
        num_samples,
        num_labels,
        True,
        4,
        device=device,
        resample=False,
        seed=seed,
    )

    module = copy.deepcopy(model)
    head_importance_prunning(
        module, model_config, positive_samples, concern, head_pruning_ratio
    )
    print(f"Evaluate the pruned model {concern}")

    result = evaluate_model(module, model_config, test_dataloader)
    similar(
        model,
        module,
        valid_dataloader,
        0,
        num_samples,
        num_labels,
        device=device,
        seed=seed,
    )

Total heads to prune: 8
head importance list
 tensor([[2.6363e-03, 2.7927e-03, 2.5726e-03, 2.5507e-03],
        [2.4888e-03, 3.2295e-03, 2.9315e-03, 2.9527e-03],
        [2.2624e-03, 2.7893e-03, 2.3855e-03, 2.3905e-03],
        [3.9015e+00, 3.7453e+00, 3.7032e+00, 4.2035e+00]], device='cuda:0')
head importance list
 tensor([[2.5013e-03, 2.7075e-03, 2.5923e-03, 2.6122e-03],
        [7.7306e-04, 3.1244e-03, 2.7641e-03, 2.8374e-03],
        [4.5097e-04, 2.4586e-03, 4.7233e-04, 4.6920e-04],
        [4.0608e+00, 4.0795e+00, 3.8790e+00, 4.2157e+00]], device='cuda:0')
{(2, 1), (0, 0), (0, 3), (2, 0), (2, 3), (0, 2), (2, 2), (1, 0)}
Evaluate the pruned model 0


Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.3812
Precision: 0.6402, Recall: 0.5685, F1-Score: 0.5855
              precision    recall  f1-score   support

           0       0.44      0.57      0.50      2941
           1       0.64      0.45      0.53      2997
           2       0.69      0.60      0.64      3016
           3       0.28      0.68      0.40      2978
           4       0.80      0.62      0.70      3017
           5       0.88      0.62      0.73      3004
           6       0.63      0.41      0.49      3037
           7       0.59      0.58      0.58      3026
           8       0.68      0.57      0.62      2997
           9       0.77      0.58      0.66      2987

    accuracy                           0.57     30000
   macro avg       0.64      0.57      0.59     30000
weighted avg       0.64      0.57      0.59     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.4254
Precision: 0.6234, Recall: 0.5478, F1-Score: 0.5584
              precision    recall  f1-score   support

           0       0.39      0.62      0.48      2941
           1       0.65      0.46      0.54      2997
           2       0.64      0.65      0.65      3016
           3       0.31      0.64      0.42      2978
           4       0.75      0.70      0.72      3017
           5       0.90      0.43      0.58      3004
           6       0.64      0.38      0.48      3037
           7       0.45      0.58      0.51      3026
           8       0.70      0.50      0.58      2997
           9       0.80      0.50      0.62      2987

    accuracy                           0.55     30000
   macro avg       0.62      0.55      0.56     30000
weighted avg       0.62      0.55      0.56     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

...

Loss: 1.4431
Precision: 0.6203, Recall: 0.5423, F1-Score: 0.5507
              precision    recall  f1-score   support

           0       0.38      0.62      0.47      2941
           1       0.70      0.40      0.51      2997
           2       0.65      0.64      0.65      3016
           3       0.30      0.63      0.41      2978
           4       0.71      0.76      0.73      3017
           5       0.87      0.41      0.56      3004
           6       0.62      0.40      0.49      3037
           7       0.46      0.57      0.51      3026
           8       0.67      0.56      0.61      2997
           9       0.83      0.43      0.57      2987

    accuracy                           0.54     30000
   macro avg       0.62      0.54      0.55     30000
weighted avg       0.62      0.54      0.55     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.3694
Precision: 0.6312, Recall: 0.5685, F1-Score: 0.5774
              precision    recall  f1-score   support

           0       0.39      0.60      0.47      2941
           1       0.66      0.45      0.54      2997
           2       0.68      0.59      0.63      3016
           3       0.31      0.65      0.42      2978
           4       0.79      0.67      0.72      3017
           5       0.82      0.75      0.79      3004
           6       0.65      0.39      0.49      3037
           7       0.53      0.63      0.57      3026
           8       0.73      0.35      0.48      2997
           9       0.75      0.59      0.66      2987

    accuracy                           0.57     30000
   macro avg       0.63      0.57      0.58     30000
weighted avg       0.63      0.57      0.58     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.3998
Precision: 0.6282, Recall: 0.5631, F1-Score: 0.5771
              precision    recall  f1-score   support

           0       0.42      0.60      0.49      2941
           1       0.66      0.42      0.51      2997
           2       0.69      0.60      0.64      3016
           3       0.29      0.66      0.40      2978
           4       0.77      0.67      0.72      3017
           5       0.85      0.57      0.68      3004
           6       0.62      0.41      0.49      3037
           7       0.58      0.50      0.54      3026
           8       0.66      0.62      0.64      2997
           9       0.75      0.59      0.66      2987

    accuracy                           0.56     30000
   macro avg       0.63      0.56      0.58     30000
weighted avg       0.63      0.56      0.58     30000

